# Focal-plane field atlas: Gaussian, LG, cylindrical-vector and full-Poincaré beams
### Richards–Wolf vectorial diffraction • method: **five Bessel integrals** • runtime: **NumPy / SciPy CPU**

A self-contained research and teaching notebook for the ideal aplanatic objective. Every example calculates the **complex** focal-plane components $E_x(x,y,0),E_y(x,y,0),E_z(x,y,0)$, plots six intensity/phase maps, and constructs an **8×8 grid of true 3D polarization ellipses** in PyVista over the total intensity pattern.

**Organization**  
1. Gaussian ($\ell=0$): H/X, V/Y, diagonal D, antidiagonal A, R and L circular.  
2. LG$_0^{+1}$ and LG$_0^{-1}$: six input Jones states **for each charge**.  
3. Four cylindrical vector beams: radial, azimuthal, hyperbolic-x, hyperbolic-y.  
4. Full-Poincaré beam: star, monstar, lemon (thesis Eq. 10 convention).

**Source:** M. F. Ferrer-García, PhD thesis, Chapter 2 and Appendix A; thesis-associated 2020 Optics Express article, cylindrical-vector Eq. (9), full-Poincaré Eq. (10). The four chosen CV examples are an illustrative basis of the cylindrical vector family, not four exclusive classes.

**Conventions:** harmonic time dependence $\mathrm{Re}[\mathbf E e^{-i\omega t}]$, propagation $e^{+ikz}$, pupil ray map $r_p=f\sin\theta$, input $E_z^{\mathrm{in}}=0$. $\lambda_0$ denotes vacuum wavelength. R/L use the explicitly declared Jones vectors below; nomenclature differs among optics communities.

**Performance:** complex128; JAX variants require an accelerator by default and transfer completed results to the CPU for plotting. All PyVista plots are produced by running the notebook; screenshots require PyVista + an off-screen rendering backend. The focus is on physics and reproducibility, not precomputed example images.

## 0. Installation and numerical parameters

**Run the next setup cell once** before the solver cells. It installs only missing Python packages, checks out the matching GitHub branch when required by Colab, configures modern PyVista/VTK off-screen rendering, and imports MathTubes. If your environment lacks an OpenGL backend, use Mesa/Xvfb as a system-specific fallback. This setup supersedes the earlier multiple installation/clone cells.\n\nWe choose $\mathrm{NA}=0.85<n=1$, wavelength units $\lambda_0=1$, and field-of-view $\sim 8\lambda_0$. These are examples; vary resolution, pupil cutoff and window independently in validation. A finite FFT grid produces a periodic approximation, while the radial integral solver interpolates sampled radial integrals. Neither is exact at finite resolution.
**Display window:** $-1.5\lambda_0\le x,y\le1.5\lambda_0$. The FFT solver retains a larger computational window for angular-spectrum sampling and crops only the **visualization**. All coordinates are expressed in units of vacuum wavelength $\lambda_0$.\n

In [ ]:
# 0. Colab/local setup — dependencies, branch checkout, PyVista and MathTubes.
# Safe to rerun. Modern PyVista/VTK can use off-screen rendering without pv.start_xvfb().
import os
import sys
import subprocess
import importlib.util
from pathlib import Path

BRANCH = "main"
REPO_URL = "https://github.com/mffg1993/TightFocusingSolutions.git"
REPO_ROOT = Path("/content/TightFocusingSolutions") if Path("/content").exists() else Path.cwd()

# Install only packages that are actually missing; retain SciPy/Matplotlib for numerics.
packages = {"numpy": "numpy", "scipy": "scipy", "matplotlib": "matplotlib",
            "pyvista": "pyvista", "vtk": "vtk", "PIL": "pillow"}
missing = [dist for module,dist in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

# Colab opens a notebook without downloading the other files in its GitHub branch.
candidates = [Path.cwd(), Path.cwd().parent, REPO_ROOT]
REPO_ROOT = next((p for p in candidates
                  if (p / "visualization" / "mathtubes.py").is_file()), REPO_ROOT)
if not (REPO_ROOT / "visualization" / "mathtubes.py").is_file():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"{REPO_ROOT} is nonempty but not a complete repository checkout.")
    subprocess.run(["git", "clone", "--depth", "1", "--single-branch",
                    "--branch", BRANCH, REPO_URL, str(REPO_ROOT)], check=True)
else:
    if (REPO_ROOT / ".git").is_dir():
        actual = subprocess.check_output(
            ["git", "-C", str(REPO_ROOT), "branch", "--show-current"], text=True).strip()
        if actual == BRANCH:
            subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"],
                           check=False)  # Keep already-edited notebook work safe.
        else:
            print(f"Note: companion files are from branch {actual!r}, expected {BRANCH!r}.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pyvista as pv
pv.OFF_SCREEN = True
pv.set_jupyter_backend("static")  # Colab-friendly static output.
from visualization.mathtubes import plot_field_ellipses
print("✓ Ready:", BRANCH, "— PyVista", pv.__version__, "—", REPO_ROOT)
# If an older Colab runtime lacks a working off-screen OpenGL backend, install
# system Mesa/Xvfb with apt-get in a separate cell. No deprecated start_xvfb call.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import jv
from scipy.integrate import simpson
from functools import lru_cache
from IPython.display import display, Image
NA=0.85; n_medium=1.0; lambda0=1.0; f=1.0
k=2*np.pi*n_medium/lambda0; k0=2*np.pi/lambda0
alpha=np.arcsin(NA/n_medium)
N = 121 if "integral" in "integral_cpu" else 256
dx = 3.0/(N-1) if "integral" in "integral_cpu" else 0.12
x=(np.arange(N)-(N//2 if "integral".startswith("fft") else (N-1)/2))*dx
X,Y=np.meshgrid(x,x,indexing="xy")
RHO=np.hypot(X,Y); PHI=np.arctan2(Y,X)
# Keep these independent so backend dispatch does not silently change the optical setup.
print(f"method=integral_cpu, NA={NA}, lambda0={lambda0}, N={N}, dx={dx}, extent={x[0]:.2f}...{x[-1]:.2f}")
if NA>=n_medium: raise ValueError("Require NA<n for this example.")
# Scale of the displayed focal region (in vacuum-wavelength units).
PLOT_HALF_WIDTH = 1.5
print(f"Displayed focal window: ±{PLOT_HALF_WIDTH} λ0; computational grid N={N}, dx={dx:.5g} λ0")


## 0A. Explicit Jones polarization conventions and beam decomposition

This notebook uses `R=(xhat - i yhat)/sqrt(2)`, `L=(xhat + i yhat)/sqrt(2)` with $\exp(-i\omega t)$. They describe opposite rotation senses, but authors may reverse the labels depending on their viewing direction. Define $D=(H+V)/\sqrt2$ and $A=(H-V)/\sqrt2$.

Every input is a **linear combination of scalar LG-like pupil modes** $h_{\ell}(q)e^{i\ell\phi}$ multiplying a complex Jones vector. Here $q=\sin\theta/\sin\alpha$ and $h_{\ell}(q)=q^{|\ell|}e^{-(q/\mathrm{fill})^2}$, with $\mathrm{fill}=1$.

**Cylindrical construction (thesis Eq. 9):** equal-weight opposite-helicity modes with charges $\pm1$. Reversing charge assignment gives the two hyperbolic counterparts. **Full-Poincaré construction (thesis Eq. 10):** $\cos\alpha_p(\cos\beta\,LG_{+1}-\sin\beta\,LG_{-1})\mathbf e_R+e^{i\delta}\sin\alpha_p\,LG_{0}\mathbf e_L$, with $\alpha_p=\pi/4$ and $\delta=0$. The labels star/monstar/lemon follow the thesis's $\beta=0,\pi/4,\pi/2$ parameterization. A definitive post-focusing classification additionally requires computing polarization-line indices, since high-NA focusing can modify the local texture.

In [ ]:
SQ2=np.sqrt(2.0)
JONES={
 "X / H":np.array([1,0],complex),
 "Y / V":np.array([0,1],complex),
 "D (+45°)":np.array([1,1],complex)/SQ2,
 "A (-45°)":np.array([1,-1],complex)/SQ2,
 "R":np.array([1,-1j],complex)/SQ2,
 "L":np.array([1,1j],complex)/SQ2,
}
def one_mode(ell,jones,weight=1):
    return (int(ell),complex(jones[0])*weight,complex(jones[1])*weight)
def gaussian(jones): return [one_mode(0,jones)]
def lg(ell,jones): return [one_mode(ell,jones)]
def cylindrical(kind):
    R,L=JONES["R"],JONES["L"]
    if kind=="Radial": return [one_mode(+1,R,1/SQ2),one_mode(-1,L,1/SQ2)]
    if kind=="Azimuthal": return [one_mode(+1,R,1/SQ2),one_mode(-1,L,-1/SQ2)]
    if kind=="Hyperbolic X": return [one_mode(-1,R,1/SQ2),one_mode(+1,L,1/SQ2)]
    if kind=="Hyperbolic Y": return [one_mode(-1,R,1/SQ2),one_mode(+1,L,-1/SQ2)]
    raise ValueError(kind)
def poincare(beta):
    R,L=JONES["R"],JONES["L"]
    return [one_mode(+1,R,np.cos(beta)/SQ2),one_mode(-1,R,-np.sin(beta)/SQ2),
            one_mode(0,L,1/SQ2)]
def pupil_mode(q,phi,ell):
    return (q**abs(ell))*np.exp(-q*q)*np.exp(1j*ell*phi)
print("Input modes initialized; each case is a list of (charge, Jones-x, Jones-y).")

## 0B. Five-integral numerical backend

The analytic azimuthal integration produces Bessel orders $\ell$, $\ell\pm1$, and $\ell\pm2$. The **longitudinal** orders $\ell\pm1$ include an additional $\sin\theta$ in their $\theta$ integrands, as required by the third row of the lens matrix. The code below uses exactly this corrected thesis convention.

The radial integrals depend only on $\rho$, so they are evaluated on a dense radial grid **once per charge**, then interpolated to the Cartesian plotting grid. This is the thesis-based approach and avoids evaluating Bessel functions at every pixel. Approximation errors come from angular quadrature and radial interpolation.

**CPU variant:** SciPy `jv` and Simpson quadrature evaluate the radial Bessel integrals. This follows the thesis-integral branch (not the FFT method).


In [ ]:
print("CPU Bessel backend: SciPy.special.jv + Simpson quadrature")

In [ ]:
def integral(values,theta): return simpson(values,x=theta,axis=1)
radial_nodes=np.linspace(0,float(np.max(RHO)),251)
theta=np.linspace(0,alpha,401)
def make_basis(ell):
    ell=int(ell)
    T=theta[None,:]; U=radial_nodes[:,None]
    q=np.sin(T)/np.sin(alpha)
    g=np.sqrt(np.cos(T))*np.sin(T)*(q**abs(ell))*np.exp(-q*q)
    arg=k*U*np.sin(T)
    ct=np.cos(T); st=np.sin(T)
    b1=integral(g*(1+ct)*jv(ell,arg),theta)
    b2=integral(g*(1-ct)*jv(ell+2,arg),theta)
    b3=integral(g*(1-ct)*jv(ell-2,arg),theta)
    b4=integral(g*st*jv(ell+1,arg),theta)
    b5=integral(g*st*jv(ell-1,arg),theta)
    fac=-np.pi*(1j**(ell+1))*f/(lambda0/n_medium)
    flong=np.pi*(1j**ell)*f/(lambda0/n_medium)
    def interp(a):
        a=np.asarray(a); r=RHO.ravel()
        return (np.interp(r,radial_nodes,a.real)+1j*np.interp(r,radial_nodes,a.imag)).reshape(RHO.shape)
    I1=fac*interp(b1)*np.exp(1j*ell*PHI)
    I2=fac*interp(b2)*np.exp(1j*(ell+2)*PHI)
    I3=fac*interp(b3)*np.exp(1j*(ell-2)*PHI)
    I4=flong*interp(b4)*np.exp(1j*(ell+1)*PHI)
    I5=flong*interp(b5)*np.exp(1j*(ell-1)*PHI)
    # Basis output [input x/y, focal Ex/Ey/Ez, y, x].
    F_x=np.stack([(2*I1+I2+I3)/2,(I2-I3)/(2j),I5-I4])
    F_y=np.stack([(I2-I3)/(2j),(2*I1-I2-I3)/2,1j*(I4+I5)])
    return np.stack([F_x,F_y])
@lru_cache(None)
def cached_basis(ell): return make_basis(ell)
def calculate(terms):
    E=np.zeros((3,N,N),complex)
    for ell,ax,ay in terms:
        B=cached_basis(ell)
        E+=ax*B[0]+ay*B[1]
    return E


## 0C. Physics checks and visualization tools

The polarization ellipse at each point is the trajectory
$\mathbf r_\mathrm{ellipse}(t)=\mathrm{Re}[\mathbf E(x,y,0)e^{-it}]=\mathrm{Re}\mathbf E\cos t+\mathrm{Im}\mathbf E\sin t$.
This is a **true 3D ellipse** because the focused field has a nonzero $E_z$.

The focal-plane display window is **±1.5 vacuum wavelengths** on both transverse axes. The sample coordinates form an **8×8** grid, rendered with the user-supplied MathTubes `tube(points, radius, closed=True)` API. The line coordinates are calculated from the full complex $E_x,E_y,E_z$ field; they are not synthetic ellipse illustrations. Ellipses are normalized *individually* for visibility; read the intensity background to determine amplitude. Only exact/near-zero field locations are omitted, since polarization is then undefined. The background plane is colored by normalized total intensity. Ellipse hue encodes the sign of $\mathrm{Im}(E_xE_y^*)$ **only as a transverse handedness proxy**, not a complete classification of three-dimensional handedness.

The plotted scalar phases are the arguments of the **complex Cartesian components**, not a uniquely defined phase for the total vector intensity. Low-amplitude pixels have undefined phase and are masked on each component plot. Nothing is modified in the complex field before ellipse reconstruction.\n\n**Source of rendering machinery:** user-provided `MathTubes_Colab.ipynb`, adapted into `visualization/mathtubes.py`. The module preserves `curve_mesh`, `tube`, `plot_tubes`, closed-curve handling and PBR lighting. The additional `plot_field_ellipses` adapter draws the intensity map and places the measured-field ellipse tubes at an 8×8 sample grid.\n

**Figure organization:** each example has separate calculation + 2D intensity/HSV-phase and PyVista polarization visualization cells. The 3D MathTubes plot uses thicker tube radii and a vertical intensity colorbar; axes are labeled in units of $\lambda_0$.\n

In [ ]:
def maps(E, title):
    """Intensity and HSV-phase maps of all three complex Cartesian fields at z=0."""
    I = np.abs(E)**2
    total = np.sum(I, axis=0)
    phase_cmap = plt.get_cmap("hsv").copy()
    phase_cmap.set_bad(color="#d6d6d6")
    fig, axs = plt.subplots(2, 3, figsize=(12.5, 7.6), constrained_layout=True)
    for j, label in enumerate(("Ex", "Ey", "Ez")):
        ax_int, ax_phase = axs[0,j], axs[1,j]
        img_I = ax_int.imshow(I[j], origin="lower",
                             extent=[x[0], x[-1], x[0], x[-1]],
                             cmap="inferno", interpolation="nearest")
        ax_int.set_title(r"$|" + label + r"|^2$")
        phase = np.ma.array(np.angle(E[j]),
                            mask=I[j] < max(1e-15, 1e-5*np.max(I[j])))
        img_phase = ax_phase.imshow(phase, origin="lower",
                                   extent=[x[0], x[-1], x[0], x[-1]],
                                   cmap=phase_cmap, vmin=-np.pi, vmax=np.pi,
                                   interpolation="nearest")
        ax_phase.set_title(r"$\\arg(" + label + r")$")
        for ax in (ax_int, ax_phase):
            ax.set_xlim(-PLOT_HALF_WIDTH, PLOT_HALF_WIDTH)
            ax.set_ylim(-PLOT_HALF_WIDTH, PLOT_HALF_WIDTH)
            ax.set_aspect("equal")
            ax.set_xlabel(r"$x/\\lambda_0$")
            ax.set_ylabel(r"$y/\\lambda_0$")
        cbar=fig.colorbar(img_phase, ax=ax_phase, fraction=0.046, pad=0.03)
        cbar.set_ticks([-np.pi,0,np.pi], labels=[r"$-\\pi$", "0", r"$\\pi$"])
    fig.suptitle(title + r" — focal plane $z=0$", fontsize=14)
    plt.show()
    plt.close(fig)
    print(f"Peak |E|²: {total.max():.6g}; longitudinal share: "
          f"{I[2].sum()/max(total.sum(),1e-30):.4f}")


In [ ]:
def pyvista_ellipses(E, title, grid_points=8, off_screen=True):
    """Render calculated 3D polarization ellipses using the submitted MathTubes API.

    Source: MathTubes_Colab.ipynb, adapted in visualization/mathtubes.py.
    The 8x8 ellipse centerlines use Re[E exp(-i omega t)], including Ez.
    """
    from pathlib import Path
    import sys
    roots = (REPO_ROOT, Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent)
    root = next((r for r in roots if (r / "visualization" / "mathtubes.py").exists()), None)
    if root is None:
        raise FileNotFoundError(
            "Missing visualization/mathtubes.py. Run this notebook from the cloned "
            "TightFocusingSolutions repository root or its tutorials/ directory.")
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    try:
        from visualization.mathtubes import plot_field_ellipses
    except ImportError as exc:
        raise ImportError("Install PyVista with %pip install pyvista and configure "
                          "an OpenGL/Xvfb backend for Colab.") from exc
    # The helper includes the intensity plane, MathTubes closed tube meshes,
    # PBR material, anti-aliasing, light kit, and the 8x8 sampling grid.
    plotter = plot_field_ellipses(
        E, x, x, ngrid=grid_points, roi=PLOT_HALF_WIDTH,
        tube_radius_fraction=0.024,
        off_screen=off_screen, size=(1100, 900), show=True)
    return plotter

def show_case(title,terms,render_3d=True):
    E=calculate(terms)
    assert E.shape==(3,N,N) and np.all(np.isfinite(E))
    maps(E,title)
    if render_3d: pyvista_ellipses(E,title)
    return E


## 1. Gaussian beam, $LG_0^{0}$
Six polarization states are constructed from the **same scalar Gaussian pupil**. An x/y polarized Gaussian produces a longitudinal focal component after high-NA focusing. For circular illumination the local focal polarization texture is not generally a uniform transverse circle. The field is always computed from the *actual lens operator*, not a paraxial Jones-vector plot.

### 1.1. Gaussian — X / H
Input: $LG_0^0$ with Jones state **X / H**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_1 = calculate(gaussian(JONES["X / H"]))
maps(E_gauss_1, "Gaussian X / H")

In [ ]:
pyvista_ellipses(E_gauss_1, "Gaussian X / H")

### 1.2. Gaussian — Y / V
Input: $LG_0^0$ with Jones state **Y / V**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_2 = calculate(gaussian(JONES["Y / V"]))
maps(E_gauss_2, "Gaussian Y / V")

In [ ]:
pyvista_ellipses(E_gauss_2, "Gaussian Y / V")

### 1.3. Gaussian — D (+45°)
Input: $LG_0^0$ with Jones state **D (+45°)**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_3 = calculate(gaussian(JONES["D (+45°)"]))
maps(E_gauss_3, "Gaussian D (+45°)")

In [ ]:
pyvista_ellipses(E_gauss_3, "Gaussian D (+45°)")

### 1.4. Gaussian — A (-45°)
Input: $LG_0^0$ with Jones state **A (-45°)**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_4 = calculate(gaussian(JONES["A (-45°)"]))
maps(E_gauss_4, "Gaussian A (-45°)")

In [ ]:
pyvista_ellipses(E_gauss_4, "Gaussian A (-45°)")

### 1.5. Gaussian — R
Input: $LG_0^0$ with Jones state **R**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_5 = calculate(gaussian(JONES["R"]))
maps(E_gauss_5, "Gaussian R")

In [ ]:
pyvista_ellipses(E_gauss_5, "Gaussian R")

### 1.6. Gaussian — L
Input: $LG_0^0$ with Jones state **L**. Compare the transverse intensity redistribution and the emergence of $E_z$; inspect the ellipse grid at sites away from the center as well as on-axis.

In [ ]:
E_gauss_6 = calculate(gaussian(JONES["L"]))
maps(E_gauss_6, "Gaussian L")

In [ ]:
pyvista_ellipses(E_gauss_6, "Gaussian L")

## 2.A. Laguerre–Gaussian beam $LG_0^{+1}$
The scalar pupil carries azimuthal phase $e^{i\ell\phi}$ with $\ell=1$. Spin-orbit coupling mixes azimuthal orders in the focal Cartesian components (the five-integral approach explicitly shows orders $\ell,\ell\pm1,\ell\pm2$). Examine component phases, not only doughnut-shaped intensity.

### LG charge 1, polarization X / H
The incident field is a single $p=0$ vortex pupil term with a uniform **X / H** Jones vector.

In [ ]:
E_lg_plus_1 = calculate(lg(1,JONES["X / H"]))
maps(E_lg_plus_1, "LG ℓ=1: X / H")

In [ ]:
pyvista_ellipses(E_lg_plus_1, "LG ℓ=1: X / H")

### LG charge 1, polarization Y / V
The incident field is a single $p=0$ vortex pupil term with a uniform **Y / V** Jones vector.

In [ ]:
E_lg_plus_2 = calculate(lg(1,JONES["Y / V"]))
maps(E_lg_plus_2, "LG ℓ=1: Y / V")

In [ ]:
pyvista_ellipses(E_lg_plus_2, "LG ℓ=1: Y / V")

### LG charge 1, polarization D (+45°)
The incident field is a single $p=0$ vortex pupil term with a uniform **D (+45°)** Jones vector.

In [ ]:
E_lg_plus_3 = calculate(lg(1,JONES["D (+45°)"]))
maps(E_lg_plus_3, "LG ℓ=1: D (+45°)")

In [ ]:
pyvista_ellipses(E_lg_plus_3, "LG ℓ=1: D (+45°)")

### LG charge 1, polarization A (-45°)
The incident field is a single $p=0$ vortex pupil term with a uniform **A (-45°)** Jones vector.

In [ ]:
E_lg_plus_4 = calculate(lg(1,JONES["A (-45°)"]))
maps(E_lg_plus_4, "LG ℓ=1: A (-45°)")

In [ ]:
pyvista_ellipses(E_lg_plus_4, "LG ℓ=1: A (-45°)")

### LG charge 1, polarization R
The incident field is a single $p=0$ vortex pupil term with a uniform **R** Jones vector.

In [ ]:
E_lg_plus_5 = calculate(lg(1,JONES["R"]))
maps(E_lg_plus_5, "LG ℓ=1: R")

In [ ]:
pyvista_ellipses(E_lg_plus_5, "LG ℓ=1: R")

### LG charge 1, polarization L
The incident field is a single $p=0$ vortex pupil term with a uniform **L** Jones vector.

In [ ]:
E_lg_plus_6 = calculate(lg(1,JONES["L"]))
maps(E_lg_plus_6, "LG ℓ=1: L")

In [ ]:
pyvista_ellipses(E_lg_plus_6, "LG ℓ=1: L")

## 2.B. Laguerre–Gaussian beam $LG_0^{-1}$
The scalar pupil carries azimuthal phase $e^{i\ell\phi}$ with $\ell=-1$. Spin-orbit coupling mixes azimuthal orders in the focal Cartesian components (the five-integral approach explicitly shows orders $\ell,\ell\pm1,\ell\pm2$). Examine component phases, not only doughnut-shaped intensity.

### LG charge -1, polarization X / H
The incident field is a single $p=0$ vortex pupil term with a uniform **X / H** Jones vector.

In [ ]:
E_lg_minus_1 = calculate(lg(-1,JONES["X / H"]))
maps(E_lg_minus_1, "LG ℓ=-1: X / H")

In [ ]:
pyvista_ellipses(E_lg_minus_1, "LG ℓ=-1: X / H")

### LG charge -1, polarization Y / V
The incident field is a single $p=0$ vortex pupil term with a uniform **Y / V** Jones vector.

In [ ]:
E_lg_minus_2 = calculate(lg(-1,JONES["Y / V"]))
maps(E_lg_minus_2, "LG ℓ=-1: Y / V")

In [ ]:
pyvista_ellipses(E_lg_minus_2, "LG ℓ=-1: Y / V")

### LG charge -1, polarization D (+45°)
The incident field is a single $p=0$ vortex pupil term with a uniform **D (+45°)** Jones vector.

In [ ]:
E_lg_minus_3 = calculate(lg(-1,JONES["D (+45°)"]))
maps(E_lg_minus_3, "LG ℓ=-1: D (+45°)")

In [ ]:
pyvista_ellipses(E_lg_minus_3, "LG ℓ=-1: D (+45°)")

### LG charge -1, polarization A (-45°)
The incident field is a single $p=0$ vortex pupil term with a uniform **A (-45°)** Jones vector.

In [ ]:
E_lg_minus_4 = calculate(lg(-1,JONES["A (-45°)"]))
maps(E_lg_minus_4, "LG ℓ=-1: A (-45°)")

In [ ]:
pyvista_ellipses(E_lg_minus_4, "LG ℓ=-1: A (-45°)")

### LG charge -1, polarization R
The incident field is a single $p=0$ vortex pupil term with a uniform **R** Jones vector.

In [ ]:
E_lg_minus_5 = calculate(lg(-1,JONES["R"]))
maps(E_lg_minus_5, "LG ℓ=-1: R")

In [ ]:
pyvista_ellipses(E_lg_minus_5, "LG ℓ=-1: R")

### LG charge -1, polarization L
The incident field is a single $p=0$ vortex pupil term with a uniform **L** Jones vector.

In [ ]:
E_lg_minus_6 = calculate(lg(-1,JONES["L"]))
maps(E_lg_minus_6, "LG ℓ=-1: L")

In [ ]:
pyvista_ellipses(E_lg_minus_6, "LG ℓ=-1: L")

## 3. Cylindrical-vector beams — four examples from the opposite-helicity LG family
Using the superposition of opposite circular states and opposite orbital charges (thesis Eq. 9), the **radial** and **azimuthal** patterns are obtained by changing the relative phase. Swapping the charge assignments yields **hyperbolic-x** and **hyperbolic-y** vector textures. The local input Jones vector is spatially varying, but each term is a conventional $LG_0^{\ell}$ with fixed Jones polarization, so both solvers can apply superposition without new physics.

### 3.1. Radial
Inspect how the **transverse** input linear-polarization texture becomes a 3D ellipse field after focusing. Radially polarized illumination is particularly useful for generating pronounced longitudinal fields.

In [ ]:
E_cv_1 = calculate(cylindrical("Radial"))
maps(E_cv_1, "CV Radial")

In [ ]:
pyvista_ellipses(E_cv_1, "CV Radial")

### 3.2. Azimuthal
Inspect how the **transverse** input linear-polarization texture becomes a 3D ellipse field after focusing. Radially polarized illumination is particularly useful for generating pronounced longitudinal fields.

In [ ]:
E_cv_2 = calculate(cylindrical("Azimuthal"))
maps(E_cv_2, "CV Azimuthal")

In [ ]:
pyvista_ellipses(E_cv_2, "CV Azimuthal")

### 3.3. Hyperbolic X
Inspect how the **transverse** input linear-polarization texture becomes a 3D ellipse field after focusing. Radially polarized illumination is particularly useful for generating pronounced longitudinal fields.

In [ ]:
E_cv_3 = calculate(cylindrical("Hyperbolic X"))
maps(E_cv_3, "CV Hyperbolic X")

In [ ]:
pyvista_ellipses(E_cv_3, "CV Hyperbolic X")

### 3.4. Hyperbolic Y
Inspect how the **transverse** input linear-polarization texture becomes a 3D ellipse field after focusing. Radially polarized illumination is particularly useful for generating pronounced longitudinal fields.

In [ ]:
E_cv_4 = calculate(cylindrical("Hyperbolic Y"))
maps(E_cv_4, "CV Hyperbolic Y")

In [ ]:
pyvista_ellipses(E_cv_4, "CV Hyperbolic Y")

## 4. Full-Poincaré beams — star, monstar, and lemon
The thesis's Eq. (10) combines $LG_0^{+1}\mathbf e_R$, $LG_0^{-1}\mathbf e_R$, and $LG_0^0\mathbf e_L$. We hold $\alpha_p=\pi/4$ and $\delta=0$, and change the coefficient-mixing angle $\beta$.

**Important distinction:** star / monstar / lemon here name the **incident target textures in the cited thesis convention**. To label a focused field's observed topological singularity rigorously, compute polarization ellipse major-axis winding and separatrix count near the relevant C-point. The 8×8 visualization illustrates geometry but is **not** by itself a topology classifier.

### 4.1. Star ($\beta=0$)
Plot full vectorial focused components and the intensity plus 3D ellipse grid. The ellipticity and orientation in the focal plane need not match the paraxial input texture.

In [ ]:
E_fp_star = calculate(poincare(0))
maps(E_fp_star, "Full-Poincaré Star")

In [ ]:
pyvista_ellipses(E_fp_star, "Full-Poincaré Star")

### 4.2. Monstar ($\beta=\pi/4$)
Plot full vectorial focused components and the intensity plus 3D ellipse grid. The ellipticity and orientation in the focal plane need not match the paraxial input texture.

In [ ]:
E_fp_monstar = calculate(poincare(np.pi/4))
maps(E_fp_monstar, "Full-Poincaré Monstar")

In [ ]:
pyvista_ellipses(E_fp_monstar, "Full-Poincaré Monstar")

### 4.3. Lemon ($\beta=\pi/2$)
Plot full vectorial focused components and the intensity plus 3D ellipse grid. The ellipticity and orientation in the focal plane need not match the paraxial input texture.

In [ ]:
E_fp_lemon = calculate(poincare(np.pi/2))
maps(E_fp_lemon, "Full-Poincaré Lemon")

In [ ]:
pyvista_ellipses(E_fp_lemon, "Full-Poincaré Lemon")

## 5. Sanity checks, convergence, and follow-up exercises
- Confirm transverse input definitions and document whether handedness is observed along $+z$ or $-z$.
- For FFT: compare discrete FFT with a direct discrete plane-wave sum and independent Debye quadrature, not only a visually plausible plot.
- For Bessel integrals: increase `theta` and `radial_nodes` density independently; inspect phase convergence near vortex zeros.
- For GPU: record actual `jax.devices()`, compilation time, synchronized execution time, dtype, and CPU-reference complex-vector errors; no GPU speedup claim is made from these example plots.
- For polarization ellipses: the displayed 8×8 grid is a **sample** of a continuously varying vector field, not a complete measure of polarization singularities. A true 3D polarization ellipse is generally tilted out of the focal plane.
- Normalize color scales consistently across cases if a quantitative visual comparison of intensity is intended; absolute incident-mode power was **not** normalized here.
- This atlas is at **z=0 only**. Multi-plane and volume propagation belong in the separate solver tutorials.

**Topological note:** In the focal region the three-dimensional ellipse is determined by the complex vector components. The transverse polarization ellipse orientation used for C-point line classification is a different projected quantity; do not conflate these.